<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Cuckoo%20Search%20Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import math
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

# 1. LOAD & SPLIT DATA (Training and Validation sets)
data = load_breast_cancer()
X_train, X_val, y_train, y_val = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42
)

# Define Hyperparameter Search Bounds:
# [n_estimators (10 to 200), max_depth (2 to 20), min_samples_split (2 to 15)]
LB = np.array([10, 2, 2])    # Lower bounds
UB = np.array([200, 20, 15]) # Upper bounds
DIM = len(LB)

# Objective Function: Evaluate a nest (hyperparameter set)
def evaluate_fitness(nest):
    # Hyperparameters must be integers for Random Forest
    n_est = int(round(nest[0]))
    depth = int(round(nest[1]))
    min_split = int(round(nest[2]))

    model = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=depth,
        min_samples_split=min_split,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    # Return validation F1-score (We want to MAXIMIZE this)
    return f1_score(y_val, preds)

# Helper function for Lévy Flight step generation (Step 5)
def levy_flight(beta=1.5):
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma, size=DIM)
    v = np.random.normal(0, 1, size=DIM)
    step = u / (np.abs(v) ** (1 / beta))
    return step

# 2. INITIALIZE CUCKOO SEARCH PARAMETERS
n_nests = 12      # Number of candidate hyperparameter sets
pa = 0.25         # Discovery probability (abandon 25% worst nests)
max_iter = 20     # Number of optimization generations
alpha = 0.05      # Step size scaling factor

# 3. INITIALIZE POPULATION (Random nests within bounds)
nests = np.random.uniform(LB, UB, size=(n_nests, DIM))

# 4. EVALUATE INITIAL FITNESS
fitness = np.array([evaluate_fitness(nest) for nest in nests])

best_idx = np.argmax(fitness)
best_nest = nests[best_idx].copy()
best_score = fitness[best_idx]

print(f"Initial Best Validation F1-Score: {best_score:.4f}")

# 5-7. MAIN ITERATIVE PROCESS
for iteration in range(max_iter):
    # Step 5: Generate new solutions via Lévy flights
    for i in range(n_nests):
        step_size = alpha * levy_flight() * (nests[i] - best_nest)
        new_nest = nests[i] + step_size
        # Keep hyperparameters inside valid lower/upper bounds
        new_nest = np.clip(new_nest, LB, UB)

        new_fitness = evaluate_fitness(new_nest)

        # Pick a random nest j to compare against
        j = np.random.randint(0, n_nests)
        if new_fitness > fitness[j]:
            nests[j] = new_nest
            fitness[j] = new_fitness

    # Step 6: Abandon worst nests (fraction pa) and build new random ones
    num_abandon = int(pa * n_nests)
    worst_indices = np.argsort(fitness)[:num_abandon] # Lowest F1-scores

    for idx in worst_indices:
        nests[idx] = np.random.uniform(LB, UB, size=DIM)
        fitness[idx] = evaluate_fitness(nests[idx])

    # Step 7: Track global best solution
    current_best_idx = np.argmax(fitness)
    if fitness[current_best_idx] > best_score:
        best_score = fitness[current_best_idx]
        best_nest = nests[current_best_idx].copy()

    print(f"Iter {iteration+1:02d}/{max_iter} | Best Val F1-Score: {best_score:.4f}")

# 8. OUTPUT THE BEST SOLUTION
opt_n_est = int(round(best_nest[0]))
opt_depth = int(round(best_nest[1]))
opt_min_split = int(round(best_nest[2]))

final_model = RandomForestClassifier(
    n_estimators=opt_n_est, max_depth=opt_depth, min_samples_split=opt_min_split, random_state=42
).fit(X_train, y_train)

train_acc = accuracy_score(y_train, final_model.predict(X_train))
val_acc = accuracy_score(y_val, final_model.predict(X_val))

print("\n--- Optimal Hyperparameters Found ---")
print(f"n_estimators      : {opt_n_est}")
print(f"max_depth         : {opt_depth}")
print(f"min_samples_split : {opt_min_split}")
print(f"Validation F1     : {best_score:.4f}")
print(f"Train vs Val Acc  : {train_acc:.4f} vs {val_acc:.4f} (Low gap = controlled overfitting)")

Initial Best Validation F1-Score: 0.9772
Iter 01/20 | Best Val F1-Score: 0.9772
Iter 02/20 | Best Val F1-Score: 0.9772
Iter 03/20 | Best Val F1-Score: 0.9772
Iter 04/20 | Best Val F1-Score: 0.9772
Iter 05/20 | Best Val F1-Score: 0.9772
Iter 06/20 | Best Val F1-Score: 0.9772
Iter 07/20 | Best Val F1-Score: 0.9772
Iter 08/20 | Best Val F1-Score: 0.9772
Iter 09/20 | Best Val F1-Score: 0.9772
Iter 10/20 | Best Val F1-Score: 0.9772
Iter 11/20 | Best Val F1-Score: 0.9772
Iter 12/20 | Best Val F1-Score: 0.9772
Iter 13/20 | Best Val F1-Score: 0.9772
Iter 14/20 | Best Val F1-Score: 0.9772
Iter 15/20 | Best Val F1-Score: 0.9772
Iter 16/20 | Best Val F1-Score: 0.9772
Iter 17/20 | Best Val F1-Score: 0.9772
Iter 18/20 | Best Val F1-Score: 0.9772
Iter 19/20 | Best Val F1-Score: 0.9772
Iter 20/20 | Best Val F1-Score: 0.9772

--- Optimal Hyperparameters Found ---
n_estimators      : 130
max_depth         : 14
min_samples_split : 2
Validation F1     : 0.9772
Train vs Val Acc  : 1.0000 vs 0.9708 (Low ga